In [1]:
# Imports and configuration
import warnings
warnings.filterwarnings('ignore')
import re, time, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
                             davies_bouldin_score, adjusted_rand_score)
from statsmodels.tsa.stattools import coint
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm
from pykalman import KalmanFilter

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

# Data
DATA_START   = '2000-01-01'   # one buffer year so the 52-week rolling Sharpe covers all of Train
END_DATE     = '2026-01-01'
SNAPSHOT     = '2001-01-01'   # constituent snapshot date
BENCHMARK    = '^SP500TR'          # auto-adjusted, so dividends are included
RF_TICKER    = '^IRX'         # 13-week T-bill yield, annualized percent
FALLBACK_RF  = 0.0            # used only if ^IRX fails to download

# Periods
TRAIN_START, TRAIN_END = '2001-01-01', '2008-12-31'   # 8 years
VAL_START,   VAL_END   = '2009-01-01', '2016-12-31'   # 8 years
TEST_START,  TEST_END  = '2017-01-01', '2025-12-31'   # 9 years

# Clustering
ROLLING_WINDOW = 52
K_VALUES       = [2, 4, 7]
K_COINT        = 7      # pair formation uses k=7 clusters to keep the candidate pool tractable
N_CORES        = 10     # only used by the multiprocess fallback for OCP

# Strategy parameters 
ZSCORE_WINDOW  = 52
VOL_WINDOW     = 26
VOL_THRESHOLD  = 1.5
HIGH_VOL_ENTRY = 2.25
LOW_VOL_ENTRY  = 1.25
STANDARD_ENTRY = 1.75
EXIT_THRESHOLD = 0.5
STOP_LOSS      = 3.25

CANDIDATE_MULTIPLIERS = [0.60, 0.75, 0.85, 1.00, 1.15, 1.25]
ACTIVITY_FLOOR        = 0.50
COST_LEVELS_BPS       = [0, 5, 10, 20]

N_BOOT, BLOCK_SIZE, BOOT_SEED = 5000, 8, 42

# calendar version (robustness). Everything outside the windows is bull.
BEAR_WINDOWS_PRIMARY = [('2018-09-20', '2018-12-24'),
                        ('2020-02-19', '2020-03-23'),
                        ('2022-01-03', '2022-10-12')]
BEAR_WINDOWS_ALT     = [('2018-10-01', '2018-12-31'),
                        ('2020-02-19', '2020-03-23'),
                        ('2022-01-03', '2022-10-12')]

METHODS = ['DTW', 'OCP', 'CORR']
LABELS  = {'DTW': 'DTW-Clustering', 'OCP': 'OCP-Clustering',
           'CORR': 'Correlation Pairs', 'Naive': 'Naive Buy-and-Hold'}
COLORS  = {'DTW': '#1f77b4', 'OCP': '#ff7f0e', 'CORR': '#2ca02c', 'Naive': '#7f7f7f'}
TAG     = 'sp500_25y'

def n_years(a, b):
    return round((pd.Timestamp(b) - pd.Timestamp(a)).days / 365.25)

print('DTW vs OCP vs Correlation Pairs vs Naive, S&P 500 25-Year Framework')
print(f'Train: {TRAIN_START} to {TRAIN_END} ({n_years(TRAIN_START, TRAIN_END)} years)')
print(f'Val:   {VAL_START} to {VAL_END} ({n_years(VAL_START, VAL_END)} years)')
print(f'Test:  {TEST_START} to {TEST_END} ({n_years(TEST_START, TEST_END)} years)')

DTW vs OCP vs Correlation Pairs vs Naive, S&P 500 25-Year Framework
Train: 2001-01-01 to 2008-12-31 (8 years)
Val:   2009-01-01 to 2016-12-31 (8 years)
Test:  2017-01-01 to 2025-12-31 (9 years)


In [2]:
# S&P 500 constituents as of the snapshot date (point-in-time membership list)
hist_url = ('https://raw.githubusercontent.com/fja05680/sp500/master/'
            'S%26P%20500%20Historical%20Components%20%26%20Changes.csv')
hist = pd.read_csv(hist_url)
hist['date'] = pd.to_datetime(hist['date'])

target_date  = pd.Timestamp(SNAPSHOT)
snapshot_row = hist[hist['date'] <= target_date].sort_values('date').iloc[-1]
raw_tickers  = sorted(t.strip() for t in snapshot_row['tickers'].split(','))

suffix_pattern = re.compile(r'-\d{6}$')   # dataset's own delisting-date suffix
universe = sorted(set(suffix_pattern.sub('', t).replace('.', '-') for t in raw_tickers))

n_suffixed = sum(1 for t in raw_tickers if suffix_pattern.search(t))
print(f'Using snapshot date: {snapshot_row["date"].date()}')
print(f'Tickers with a delisting-date suffix in the source data: {n_suffixed}')
print(f'S&P 500 constituents as of {target_date.date()}: {len(universe)} stocks')

Using snapshot date: 2000-12-29
Tickers with a delisting-date suffix in the source data: 204
S&P 500 constituents as of 2001-01-01: 492 stocks


In [3]:
# Prices: daily download, resampled to true Wednesday closes
def to_close_frame(raw):
    close = raw['Close'] if isinstance(raw.columns, pd.MultiIndex) else raw[['Close']]
    return close if isinstance(close, pd.DataFrame) else close.to_frame()

print(f'Downloading daily prices for {len(universe)} stocks...')
raw = yf.download(universe, start=DATA_START, end=END_DATE, interval='1d',
                  auto_adjust=True, progress=True, threads=True)
daily = to_close_frame(raw)
price_data = daily.resample('W-WED').last().dropna(how='all')

before_drop = len(universe)
price_data  = price_data.ffill().dropna(axis=1)   # complete history required

TAIL_WINDOW = 10
flatlined = [t for t in price_data.columns
             if price_data[t].iloc[-TAIL_WINDOW:].nunique() == 1]
price_data = price_data.drop(columns=flatlined)

dropped_tickers = sorted(set(universe) - set(price_data.columns))
print(f'\nPrice data shape: {price_data.shape}')
print(f'Date range: {price_data.index[0].date()} to {price_data.index[-1].date()}')
print(f'Retained: {price_data.shape[1]} of {before_drop} (dropped {len(dropped_tickers)}, '
      f'of which {len(flatlined)} flatlined)')
print(f'Dropped tickers: {dropped_tickers}')

# Benchmark: S&P 500 Total Return index (dividends reinvested, no fees)
bench_raw = to_close_frame(yf.download(BENCHMARK, start=DATA_START, end=END_DATE,
                                       interval='1d', auto_adjust=True, progress=False))
bench_weekly = bench_raw.iloc[:, 0].resample('W-WED').last().reindex(price_data.index)
n_missing = int(bench_weekly.isna().sum())
print(f'\nBenchmark {BENCHMARK}: {n_missing} missing weeks before filling (should be 0)')
bench_price = bench_weekly.ffill()

try:
    irx_raw = to_close_frame(yf.download(RF_TICKER, start=DATA_START, end=END_DATE,
                                         interval='1d', auto_adjust=False, progress=False))
    irx = irx_raw.iloc[:, 0].resample('W-WED').last().reindex(price_data.index).ffill().bfill()
    rf_weekly = np.log1p(irx / 100) / 52       # weekly log risk-free rate
    print(f'T-bill loaded: mean annual yield over sample {irx.mean():.2f}%')
except Exception as e:
    rf_weekly = pd.Series(FALLBACK_RF / 52, index=price_data.index)
    print(f'^IRX download failed ({e}); using constant rf={FALLBACK_RF}')

price_data.to_csv(f'price_data_{TAG}.csv')

$JP: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AL"}}}
$LEG: possibly delisted; no price data found  (1d 2000-01-01 -> 2026-01-01) (Yahoo error = "Data doesn't exist for startDate = 946702800, endDate = 1767243600")
[                       0%                       ]  2 of 492 completed$GDT: possibly delisted; no price data found  (1d 2000-01-01 -> 2026-01-01) (Yahoo error = "Data doesn't exist for startDate = 946702800, endDate = 1767243600")
[                       1%                       ]  3 of 492 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: OAT"}}}
$AL: possibly delisted; no timezone found
$OAT: possibly delisted; no timezone found       ]  4 of 492 completed
$JCP: possibly delisted; no timezone found       ]  14 of 492 completed
$RDC: possibly delisted; no timezone found       ]  19 of 49


Price data shape: (1357, 236)
Date range: 2000-01-05 to 2025-12-31
Retained: 236 of 492 (dropped 256, of which 0 flatlined)
Dropped tickers: ['AABA', 'AAMRQ', 'ABI', 'ABKFQ', 'ABS', 'ABX', 'ACV', 'ADCT', 'AET', 'AGC', 'AGN', 'AL', 'ALTR', 'AM', 'ANDW', 'APC', 'APCC', 'ARNC', 'ASO', 'AT', 'AV', 'AVP', 'AW', 'AYE', 'AZA-A', 'BBBY', 'BBT', 'BCR', 'BDK', 'BEAM', 'BGEN', 'BGG', 'BHGE', 'BIG', 'BK', 'BLL', 'BLS', 'BMC', 'BMET', 'BMS', 'BNI', 'BOL', 'BR', 'BRCM', 'BSC', 'BUD', 'BVSN', 'CA', 'CBE', 'CBS', 'CCE', 'CCTYQ', 'CEG', 'CEN', 'CF', 'CFC', 'CGP', 'CHIR', 'CIN', 'CIT-A', 'CMA', 'CMVT', 'CNC', 'CNXT', 'COC-B', 'CPNLQ', 'CPQ', 'CPWR', 'CR', 'CTB', 'CTL', 'CTX', 'CTXS', 'CVG', 'DALRQ', 'DCNAQ', 'DELL', 'DG', 'DJ', 'DOW', 'DPHIQ', 'DYN', 'EC', 'EDS', 'EKDKQ', 'EMC', 'ENRNQ', 'ETS', 'FBF', 'FDC', 'FLR', 'FRX', 'G', 'GAS', 'GDT', 'GDW', 'GLK', 'GP', 'GPS', 'GPU', 'GR', 'GTW', 'GX', 'H', 'HCA', 'HCR', 'HES', 'HET', 'HI', 'HLT', 'HM', 'HNZ', 'HOT', 'HPC', 'HSH', 'INCLF', 'IPG', 'IR', 'JAVA', '

In [4]:
# 52-week rolling Sharpe ratio in excess of the T-bill
log_returns  = np.log(price_data / price_data.shift(1))
excess_lr    = log_returns.sub(rf_weekly, axis=0)
rolling_mean = excess_lr.rolling(ROLLING_WINDOW).mean()
rolling_std  = log_returns.rolling(ROLLING_WINDOW).std()
sharpe_data  = (rolling_mean / rolling_std * np.sqrt(52)).dropna(how='all')

train_sharpe_data = sharpe_data.loc[TRAIN_START:TRAIN_END]
bad_cols = train_sharpe_data.columns[
    train_sharpe_data.isna().any() | np.isinf(train_sharpe_data).any()].tolist()
if bad_cols:
    train_sharpe_data = train_sharpe_data.drop(columns=bad_cols)
    print(f'Dropped {bad_cols} (NaN or inf rolling Sharpe in Train)')

tickers = train_sharpe_data.columns.tolist()
n = len(tickers)
print(f'Clustering universe: {n} stocks, {n*(n-1)//2:,} pairs, '
      f'{len(train_sharpe_data)} Train weeks')

Clustering universe: 236 stocks, 27,730 pairs, 418 Train weeks


In [5]:
# DTW distance matrix (Train only)
series = [train_sharpe_data[t].values.astype(np.float64) for t in tickers]
t0 = time.time()
dtw_distance_matrix = dtw.distance_matrix_fast(series)
dtw_distance_matrix[dtw_distance_matrix == np.inf] = 0
dtw_distance_matrix = dtw_distance_matrix + dtw_distance_matrix.T
print(f'DTW distance matrix computed in {time.time()-t0:.1f} seconds, shape {dtw_distance_matrix.shape}')
np.save(f'dtw_distance_matrix_{TAG}_train.npy', dtw_distance_matrix)

DTW distance matrix computed in 3.8 seconds, shape (236, 236)


In [ ]:
def cluster_and_validate(dist, name):
    link = linkage(squareform(dist, checks=False), method='ward')
    clusters, cvi = {}, []
    for k in K_VALUES:
        clusters[k] = fcluster(link, t=k, criterion='maxclust')
        sizes = pd.Series(clusters[k]).value_counts().sort_index().tolist()
        sil = silhouette_score(dist, clusters[k], metric='precomputed')
        ch  = calinski_harabasz_score(train_sharpe_data.T, clusters[k])
        db  = davies_bouldin_score(train_sharpe_data.T, clusters[k])
        cvi.append({'k': k, 'sizes': sizes, 'silhouette': sil,
                    'calinski_harabasz': ch, 'davies_bouldin': db})
    